04_gold_weaather...

In [0]:
  import pyspark.sql.functions as F
  from pyspark.sql.functions import col

In [0]:
dbutils.widgets.text("gold_catalog", "dbr_dev")
dbutils.widgets.text("gold_schema", "artemzharkov10_gold")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

In [0]:
GOLD_HISTORY_WEATHER_CLASTERS_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_weather_clusters"
GOLD_WEATHER_CLASTER_WEIGHTS_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_weather_cluster_weights"

GOLD_OUTPUT_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_history_weather_weights"

In [0]:
df_history_weather_clasters = spark.table(GOLD_HISTORY_WEATHER_CLASTERS_TABLE)
df_clasters_weights = spark.table(GOLD_WEATHER_CLASTER_WEIGHTS_TABLE)

In [0]:
# display(df_clasters)
# display(df_clasters_weights)

In [0]:
voivodeship_eng_map = {
    "Zachodniopomorskie": "West Pomeranian",
    "Wielkopolskie": "Greater Poland",
    "Warmińsko-Mazurskie": "Warmian-Masurian",
    "Świętokrzyskie": "Holy Cross",
    "Śląskie": "Silesian",
    "Pomorskie": "Pomeranian",
    "Podlaskie": "Podlaskie",
    "Podkarpackie": "Subcarpathian",
    "Opolskie": "Opole",
    "Mazowieckie": "Masovian",
    "Małopolskie": "Lesser Poland",
    "Lubuskie": "Lubusz",
    "Lubelskie": "Lublin",
    "Łódzkie": "Łódź",
    "Kujawsko-Pomorskie": "Kuyavian-Pomeranian",
    "Dolnośląskie": "Lower Silesian"
}

df_filtered_weather_clasters = (
    df_history_weather_clasters
    .select("voivodeship", "lat", "lon", "time", "weather_claster")
    .replace(voivodeship_eng_map, subset=["voivodeship"])
)

df_filtered_clasters_weights = df_clasters_weights.select("weather_claster", "normalized_risk_index")

# display(df_filtered_weather_clasters)
# display(df_filtered_clasters_weights)


In [0]:
gf_history_wheather_weights = df_filtered_weather_clasters.join(
    df_filtered_clasters_weights,
    on=["weather_claster"],
    how="left"
)

In [0]:
# display(gf_history_wheather_weights)

In [0]:
(
    gf_history_wheather_weights.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_OUTPUT_TABLE)
)